# Scenario 1 DL4DS (Forecast Correction)
This notebook reuses the Scenario 1 data flow (IFS low-res forecast + ERA5 high-res truth) and trains DL4DS per variable.

In [1]:
import os

os.chdir('/home/jovyan/work')  # Move to climate-research-workbench root
print(f'CWD: {os.getcwd()}')

CWD: /home/jovyan/work


In [ ]:
import os
os.environ['SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL'] = 'True'

# Install DL4DS stack (run once per fresh environment)
!pip install dl4ds climetlab
!pip uninstall keras -y
!pip install "tensorflow[and-cuda]==2.15.*"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 15.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 MB 34.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 18.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 19.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 19.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.3/30.3 MB 21.1 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 20.3 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.7/661

In [2]:
# ==========================================
# Run folder + logger
# ==========================================
import json as _json
import datetime as _dt

_run_start = _dt.datetime.now()
_run_id = _run_start.strftime('%Y%m%d_%H%M%S')
RUN_DIR = os.path.join('runs', f'{_run_id}_sc1_dl4ds')
os.makedirs(RUN_DIR, exist_ok=True)
_LOG_PATH = os.path.join(RUN_DIR, 'runtime.log')

VAR_LABELS = ['U10 (m/s)', 'V10 (m/s)', 'T2m (K)', 'TP 24hr (mm)']

def _log(msg: str):
    ts = _dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(_LOG_PATH, 'a') as f:
        f.write(line + '\n')

_log(f'=== Run started (id={_run_id}) ===')
_log(f'Run dir: {RUN_DIR}')

[2026-04-02 05:51:46] === Run started (id=20260402_055146) ===
[2026-04-02 05:51:46] Run dir: runs/20260402_055146_sc1_dl4ds


In [3]:
# ==========================================
# Imports + config
# ==========================================
import numpy as np
import pandas as pd
import xarray as xr
import tensorflow as tf
import dl4ds as dds

scale = 6
lead_days = 1
lead_td = np.timedelta64(lead_days, 'D')
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

train_start_date = '2018-01-01'
train_end_date = '2021-12-31'
val_start_date = '2022-01-01'
val_end_date = '2022-06-30'
test_start_date = '2022-07-01'
test_end_date = '2022-12-31'

_config = {
    'scenario': 'scenario1-dl4ds-forecast-correction',
    'run_id': _run_id,
    'run_dir': RUN_DIR,
    'scale': scale,
    'lead_days': int(lead_days),
    'seed': int(seed),
    'train_start_date': train_start_date,
    'train_end_date': train_end_date,
    'val_start_date': val_start_date,
    'val_end_date': val_end_date,
    'test_start_date': test_start_date,
    'test_end_date': test_end_date,
    'started_at': _run_start.isoformat()
}
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    _json.dump(_config, f, indent=2)
_log('Config saved -> config.json')

2026-04-02 05:51:47.777165: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-02 05:51:47.799129: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-02 05:51:47.799140: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-02 05:51:47.799617: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-02 05:51:47.803088: I tensorflow/core/platform/cpu_feature_guar

[2026-04-02 05:51:50] Config saved -> config.json


In [4]:
# ==========================================
# Load Scenario 1 datasets
# ==========================================
ds_forecast = xr.open_dataset('data/ifs_lowres_indonesia_2018-2022.zarr')
ds_truth = xr.open_dataset('data/era5_indonesia_2018-2022.zarr')

# Lead selection
ds_forecast = ds_forecast.sel(prediction_timedelta=lead_td)

# Sort latitude for stable slicing
ds_truth = ds_truth.sortby('latitude')
ds_forecast = ds_forecast.sortby('latitude')

print('Forecast dims:', ds_forecast.dims)
print('Truth dims   :', ds_truth.dims)

Forecast dims: FrozenMappingWarningOnValuesAccess({'time': 3652, 'longitude': 41, 'latitude': 30})
Truth dims   : FrozenMappingWarningOnValuesAccess({'time': 7304, 'latitude': 181, 'longitude': 201})


In [5]:
# ==========================================
# Spatial crop using SC1 logic (target 24x32 -> 144x192)
# ==========================================
tr_lons = ds_truth.longitude.values
tr_lats = ds_truth.latitude.values
fc_lons = ds_forecast.longitude.values
fc_lats = ds_forecast.latitude.values

valid_lons = fc_lons[(fc_lons >= tr_lons.min()) & (fc_lons <= tr_lons.max())]
valid_lats = fc_lats[(fc_lats >= tr_lats.min()) & (fc_lats <= tr_lats.max())]

lon_start = valid_lons[0]
lat_start = valid_lats[0]

lon_start_idx = np.argmin(np.abs(tr_lons - lon_start))
lat_start_idx = np.argmin(np.abs(tr_lats - lat_start))

avail_lon = len(tr_lons) - lon_start_idx
avail_lat = len(tr_lats) - lat_start_idx
max_fc_lon = min(avail_lon // scale, 32)
max_fc_lat = min(avail_lat // scale, 24)

fc_lon_start_idx = np.argmin(np.abs(fc_lons - lon_start))
fc_lat_start_idx = np.argmin(np.abs(fc_lats - lat_start))

ds_fc = ds_forecast.isel(
    longitude=slice(fc_lon_start_idx, fc_lon_start_idx + max_fc_lon),
    latitude=slice(fc_lat_start_idx, fc_lat_start_idx + max_fc_lat)
)

LOW_LON = len(ds_fc.longitude)
LOW_LAT = len(ds_fc.latitude)

ds_tr = ds_truth.isel(
    longitude=slice(lon_start_idx, lon_start_idx + LOW_LON * scale),
    latitude=slice(lat_start_idx, lat_start_idx + LOW_LAT * scale)
)

print(f'Low-res grid  : {LOW_LAT} x {LOW_LON}')
print(f'High-res grid : {len(ds_tr.latitude)} x {len(ds_tr.longitude)}')

Low-res grid  : 24 x 32
High-res grid : 144 x 192


In [6]:
# ==========================================
# Temporal alignment
# ==========================================
valid_time = ds_fc.time + lead_td
common_times = np.intersect1d(valid_time.values, ds_tr.time.values)

ds_fc = ds_fc.assign_coords(valid_time=valid_time)
ds_fc = ds_fc.sel(valid_time=common_times)
ds_fc = ds_fc.assign_coords(time=ds_fc.valid_time).drop_vars('valid_time')
ds_tr_aligned = ds_tr.sel(time=common_times)

print('Common timesteps:', len(common_times))
print('Forecast and truth time aligned:', np.array_equal(ds_fc.time.values, ds_tr_aligned.time.values))

Common timesteps: 3650
Forecast and truth time aligned: True


In [ ]:
# ==========================================
# DL4DS training per variable
# ==========================================
VARS = [
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    '2m_temperature',
    'total_precipitation_24hr',
]

var_names = [v for v in VARS if v in ds_fc.data_vars and v in ds_tr_aligned.data_vars]
if not var_names:
    raise ValueError('No overlapping variables found between ds_fc and ds_tr_aligned.')

times = ds_fc.time.values
train_mask = (times >= np.datetime64(train_start_date)) & (times <= np.datetime64(train_end_date))
val_mask = (times >= np.datetime64(val_start_date)) & (times <= np.datetime64(val_end_date))
test_mask = (times >= np.datetime64(test_start_date)) & (times <= np.datetime64(test_end_date))

if not (train_mask.any() and val_mask.any() and test_mask.any()):
    raise ValueError('Invalid split: one of train/val/test is empty.')

ARCH_PARAMS = dict(
    n_filters=36,
    n_blocks=8,
    normalization=None,
    dropout_rate=0.0,
    dropout_variant='spatial',
    attention=False,
    activation='relu',
    localcon_layer=False,
)

trainers = {}
data = {}
norm_stats = {}

def fill_nan(da, var_name, tag):
    """Fill NaNs with variable mean using numpy mask"""
    arr = da.values.astype(np.float32, copy=True)
    mask = np.isnan(arr)
    if mask.any():
        fill_val = float(np.nanmean(arr))
        arr[mask] = fill_val
        print(f"  {tag} {var_name}: filled {int(mask.sum())} NaNs with mean={fill_val:.4f}")
    return xr.DataArray(arr, coords=da.coords, dims=da.dims, name=da.name)

for i, var in enumerate(var_names, start=1):
    print('\n' + '=' * 64)
    print(f'Training DL4DS variable {i}/{len(var_names)}: {var}')
    print('=' * 64)

    target_var = ds_tr_aligned[[var]].to_array(dim='channel').transpose('time', 'latitude', 'longitude', 'channel')
    input_var = ds_fc[[var]].to_array(dim='channel').transpose('time', 'latitude', 'longitude', 'channel')

    # Fill NaNs before normalization/training to avoid DL4DS failures.
    # This follows the same explicit numpy-mask pattern used in sc1_unet.ipynb.
    target_var = fill_nan(target_var, var, 'Y')
    input_var = fill_nan(input_var, var, 'X')

    train_idx = np.where(train_mask)[0]
    val_idx = np.where(val_mask)[0]
    test_idx = np.where(test_mask)[0]

    train_data = target_var.isel(time=train_idx).values
    mean_val = float(np.nanmean(train_data))
    std_val = float(np.nanstd(train_data))
    if std_val < 1e-8:
        std_val = 1.0

    norm_stats[var] = {'mean': mean_val, 'std': std_val}

    target_norm = (target_var - mean_val) / std_val
    input_norm = (input_var - mean_val) / std_val

    data_var = {
        'train_target': target_norm.isel(time=train_idx),
        'val_target': target_norm.isel(time=val_idx),
        'test_target': target_norm.isel(time=test_idx),
        'train_input': input_norm.isel(time=train_idx),
        'val_input': input_norm.isel(time=val_idx),
        'test_input': input_norm.isel(time=test_idx),
        'test_target_orig': target_var.isel(time=test_idx),
        'test_input_orig': input_var.isel(time=test_idx)
    }

    trainer = dds.SupervisedTrainer(
        backbone='resnet',
        upsampling='spc',
        data_train=data_var['train_target'],
        data_val=data_var['val_target'],
        data_test=data_var['test_target'],
        data_train_lr=data_var['train_input'],
        data_val_lr=data_var['val_input'],
        data_test_lr=data_var['test_input'],
        scale=scale,
        time_window=None,
        static_vars=None,
        predictors_train=None,
        predictors_val=None,
        predictors_test=None,
        interpolation='inter_area',
        patch_size=None,
        batch_size=64,
        loss='mae',
        epochs=40,
        steps_per_epoch=None,
        validation_steps=None,
        test_steps=None,
        learning_rate=(1e-3, 1e-4),
        lr_decay_after=1e4,
        early_stopping=True,
        patience=6,
        min_delta=0,
        save=False,
        save_path=None,
        show_plot=False,
        verbose=True,
        device='GPU',
        **ARCH_PARAMS
    )

    trainer.run()
    trainers[var] = trainer
    data[var] = data_var


Training DL4DS variable 1/4: 10m_u_component_of_wind
  X 10m_u_component_of_wind: filled 768 NaNs with mean=-1.4693


2026-04-02 05:52:10.622125: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-02 05:52:10.927911: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-02 05:52:10.931449: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

List of devices:
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Number of devices: 1
Global batch size: 64
--------------------------------------------------------------------------------
Starting time: 2026-04-02 05:52:10
--------------------------------------------------------------------------------
Model: "resnet_spc"
______________________________________________________________________________________________________________________________________________________
 Layer (type)                                Output Shape                                 Param #        Connected to                                 
 input_1 (InputLayer)                        [(None, None, None, 1)]                      0              []                                           
                                                                                                                                                      
 conv2d (Conv2D)                             (None, 

2026-04-02 05:52:13.659469: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2026-04-02 05:52:13.704001: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-04-02 05:52:13.962306: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-04-02 05:52:25.068217: I external/local_xla/xla/service/service.cc:168] XLA service 0x772349896ae0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-02 05:52:25.068228: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-04-02 05:52:25.071057: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775109145.125874     680 device_compiler.h:186] 

45/45 [==============================] - 68s 590ms/step - loss: 0.3329 - val_loss: 0.2353
Epoch 2/40
45/45 [==============================] - 17s 384ms/step - loss: 0.2234 - val_loss: 0.2205
Epoch 3/40
45/45 [==============================] - 17s 383ms/step - loss: 0.2143 - val_loss: 0.2153
Epoch 4/40
45/45 [==============================] - 17s 384ms/step - loss: 0.2113 - val_loss: 0.2129
Epoch 5/40
45/45 [==============================] - 17s 384ms/step - loss: 0.2086 - val_loss: 0.2114
Epoch 6/40
45/45 [==============================] - 17s 384ms/step - loss: 0.2079 - val_loss: 0.2164
Epoch 7/40
45/45 [==============================] - 17s 385ms/step - loss: 0.2069 - val_loss: 0.2106
Epoch 8/40
45/45 [==============================] - 17s 385ms/step - loss: 0.2042 - val_loss: 0.2079
Epoch 9/40
45/45 [==============================] - 17s 385ms/step - loss: 0.2024 - val_loss: 0.2070
Epoch 10/40
45/45 [==============================] - 17s 385ms/step - loss: 0.2006 - val_loss: 0.2051


In [8]:
# ==========================================
# Aggregate test metrics + bilinear baseline
# ==========================================
rows = []
baseline_rmse_by_var = {}

fc_test_lowres = ds_fc.sel(time=slice(test_start_date, test_end_date)).copy(deep=True)
tr_test_ds = ds_tr_aligned.sel(time=slice(test_start_date, test_end_date))

for v_name in var_names:
    if np.isnan(fc_test_lowres[v_name].values).any():
        fill_val = float(np.nanmean(fc_test_lowres[v_name].values))
        fc_test_lowres[v_name] = fc_test_lowres[v_name].fillna(fill_val)

for v_name in var_names:
    low_da, truth_da = xr.align(fc_test_lowres[v_name], tr_test_ds[v_name], join='inner')
    base_da = low_da.interp(latitude=truth_da.latitude, longitude=truth_da.longitude, method='linear')
    base_da, truth_da = xr.align(base_da, truth_da, join='inner')
    base_v = base_da.transpose('time', 'latitude', 'longitude').values
    true_v = truth_da.transpose('time', 'latitude', 'longitude').values
    baseline_rmse_by_var[v_name] = float(np.nanmean(np.sqrt(np.mean((base_v - true_v) ** 2, axis=0))))

print('\n' + '=' * 110)
print(f"{'Variable':<30} | {'RMSE':>8} | {'MAE':>8} | {'Bias':>8} | {'Corr':>8} | {'Baseline RMSE':>14} | {'Skill':>8}")
print('=' * 110)

for v_name in var_names:
    model = trainers[v_name].model
    mean_val = norm_stats[v_name]['mean']
    std_val = norm_stats[v_name]['std']

    pred_time = []
    true_time = []
    n_test = len(data[v_name]['test_target'].time)

    for j in range(n_test):
        x_norm = data[v_name]['test_input'].isel(time=j).values
        y_true = data[v_name]['test_target_orig'].isel(time=j).values[:, :, 0]

        y_pred_norm = model.predict(np.expand_dims(x_norm, axis=0), verbose=0)[0]
        y_pred = y_pred_norm[:, :, 0] * std_val + mean_val

        # Safety guard for swapped spatial axes.
        if y_pred.shape != y_true.shape:
            if y_pred.shape == y_true.shape[::-1]:
                y_true = y_true.T
            else:
                raise ValueError(f'Spatial mismatch for {v_name}: pred {y_pred.shape} vs true {y_true.shape}')

        pred_time.append(y_pred)
        true_time.append(y_true)

    pred_v = np.stack(pred_time, axis=0)
    true_v = np.stack(true_time, axis=0)

    rmse_grid = np.sqrt(np.mean((pred_v - true_v) ** 2, axis=0))
    mae_grid = np.mean(np.abs(pred_v - true_v), axis=0)
    bias_grid = np.mean(pred_v - true_v, axis=0)

    pred_diff = pred_v - np.mean(pred_v, axis=0)
    true_diff = true_v - np.mean(true_v, axis=0)
    cov = np.sum(pred_diff * true_diff, axis=0)
    var_pred = np.sum(pred_diff ** 2, axis=0)
    var_true = np.sum(true_diff ** 2, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        corr_grid = cov / np.sqrt(var_pred * var_true)

    rmse = float(np.nanmean(rmse_grid))
    mae = float(np.nanmean(mae_grid))
    bias = float(np.nanmean(bias_grid))
    corr = float(np.nanmean(corr_grid))

    rmse_base = baseline_rmse_by_var[v_name]
    skill = float(1.0 - (rmse / rmse_base)) if rmse_base > 1e-8 else 0.0

    print(f"{v_name:<30} | {rmse:8.4f} | {mae:8.4f} | {bias:+8.4f} | {corr:8.4f} | {rmse_base:14.4f} | {skill:+8.4f}")

    rows.append({
        'variable': v_name,
        'RMSE': rmse,
        'MAE': mae,
        'Bias': bias,
        'Corr': corr,
        'Baseline_RMSE': rmse_base,
        'Skill': skill
    })

print('=' * 110)
print('Skill > 0 means DL4DS improves over bilinear interpolation baseline.')

metrics_df = pd.DataFrame(rows).sort_values('Skill', ascending=False).reset_index(drop=True)
metrics_path = os.path.join(RUN_DIR, 'sc1_dl4ds_metrics.csv')
metrics_df.to_csv(metrics_path, index=False)
print(f'Saved: {metrics_path}')
_log(f'Metrics saved -> {metrics_path}')

_total = _dt.datetime.now() - _run_start
_log(f"=== Run finished total_elapsed={str(_total).split('.')[0]} ===")
_log(f'All outputs in: {RUN_DIR}')


Variable                       |     RMSE |      MAE |     Bias |     Corr |  Baseline RMSE |    Skill
10m_u_component_of_wind        |   1.0800 |   0.8201 |  +0.0203 |   0.8874 |         1.1679 |  +0.0753
10m_v_component_of_wind        |   1.0637 |   0.8092 |  +0.0657 |   0.8711 |         1.0916 |  +0.0256
2m_temperature                 |   0.5566 |   0.4335 |  -0.0024 |   0.8364 |         0.7312 |  +0.2389
total_precipitation_24hr       |   0.0064 |   0.0033 |  -0.0006 |   0.8146 |         0.0060 |  -0.0628
Skill > 0 means DL4DS improves over bilinear interpolation baseline.
Saved: runs/20260402_055146_sc1_dl4ds/sc1_dl4ds_metrics.csv
[2026-04-02 06:36:13] Metrics saved -> runs/20260402_055146_sc1_dl4ds/sc1_dl4ds_metrics.csv
[2026-04-02 06:36:13] === Run finished total_elapsed=0:44:27 ===
[2026-04-02 06:36:13] All outputs in: runs/20260402_055146_sc1_dl4ds
